In [3]:
import os

import numpy as np
import torch
import torch.nn as nn
from scipy.special import ive

# ============================================================
# Device selection
# Priority: CUDA -> MPS -> CPU
# ============================================================

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

Using device: mps


In [5]:
# ============================================================
# Circle dataset
# ============================================================

def sample_circle(n, device=device):
    theta = 2 * np.pi * torch.rand(n, device=device)

    z = torch.stack(
        [
            torch.cos(theta),
            torch.sin(theta),
        ],
        dim=-1,
    )

    return z


def sample_xt(n, device=device):
    z = sample_circle(n, device)

    eps = torch.randn_like(z)

    t = torch.rand(n, 1, device=device)

    x = t * z + (1 - t) * eps

    target_velocity = z - eps

    return x, t, target_velocity


def true_velocity(pts_np, t_val):
    """Closed-form velocity field for the t-marginal of the circle flow."""
    x1, x2 = pts_np[:, 0], pts_np[:, 1]
    r = np.sqrt(x1**2 + x2**2) + 1e-12
    s = max(1 - t_val, 1e-6)
    kappa = t_val * r / s**2
    A = ive(1, kappa) / (ive(0, kappa) + 1e-12)
    x_hat = pts_np / r[:, None]
    return ((A - r) / s)[:, None] * x_hat


# ============================================================
# Flow matching network
# ============================================================

class FMNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(3, 32),
            nn.SiLU(),
            nn.Linear(32, 32),
            nn.SiLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x, t):
        inp = torch.cat([x, t], dim=-1)
        return self.net(inp)

In [6]:
# ============================================================
# Train 32 randomly initialised networks, save each to disk
# ============================================================

WEIGHTS_DIR = "flow_circle_weights"
os.makedirs(WEIGHTS_DIR, exist_ok=True)

N_MODELS = 32
TOTAL_STEPS = 20000
BATCH_SIZE = 4096
LR = 1e-4


def train_one(seed, total_steps=TOTAL_STEPS, batch_size=BATCH_SIZE, lr=LR):
    torch.manual_seed(seed)

    net = FMNet().to(device)
    opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps)

    for step in range(total_steps):
        x, t, v = sample_xt(batch_size, device=device)

        pred = net(x, t)
        loss = (pred - v).pow(2).mean()

        opt.zero_grad()
        loss.backward()
        opt.step()
        scheduler.step()

    return net, loss.item()


for seed in range(N_MODELS):
    net, final_loss = train_one(seed)
    path = os.path.join(WEIGHTS_DIR, f"model_{seed:02d}.pt")
    torch.save(net.state_dict(), path)
    print(f"seed={seed:2d}  final_loss={final_loss:.6f}  saved -> {path}")

seed= 0  final_loss=0.998388  saved -> flow_circle_weights/model_00.pt
seed= 1  final_loss=1.041711  saved -> flow_circle_weights/model_01.pt
seed= 2  final_loss=1.038133  saved -> flow_circle_weights/model_02.pt
seed= 3  final_loss=0.983930  saved -> flow_circle_weights/model_03.pt
seed= 4  final_loss=1.006648  saved -> flow_circle_weights/model_04.pt
seed= 5  final_loss=1.009156  saved -> flow_circle_weights/model_05.pt
seed= 6  final_loss=0.989636  saved -> flow_circle_weights/model_06.pt
seed= 7  final_loss=1.006126  saved -> flow_circle_weights/model_07.pt
seed= 8  final_loss=1.025441  saved -> flow_circle_weights/model_08.pt
seed= 9  final_loss=1.022373  saved -> flow_circle_weights/model_09.pt
seed=10  final_loss=1.015629  saved -> flow_circle_weights/model_10.pt
seed=11  final_loss=1.008700  saved -> flow_circle_weights/model_11.pt
seed=12  final_loss=1.030553  saved -> flow_circle_weights/model_12.pt
seed=13  final_loss=0.999648  saved -> flow_circle_weights/model_13.pt
seed=1

In [8]:
# ============================================================
# Interactive residuals plot:
#   - click anywhere on the circle plot to set the evaluation point (x, y)
#   - slider for t
#   - slider for the Gaussian-fit ellipse scale (n_std)
#   - left: raw residuals, right: whitened residuals (|J|^-1 r), both on top
#     of the true density + unit circle
# ============================================================

%matplotlib widget
import glob

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import ipywidgets as widgets
from ipywidgets import VBox, HBox

WEIGHTS_DIR = "flow_circle_weights"

def _load_models(weights_dir=WEIGHTS_DIR):
    nets = []
    for path in sorted(glob.glob(os.path.join(weights_dir, "model_*.pt"))):
        net = FMNet().to(device)
        net.load_state_dict(torch.load(path, map_location=device))
        net.eval()
        nets.append(net)
    return nets


_nets = _load_models()
if not _nets:
    raise RuntimeError(f"No model weights found in '{WEIGHTS_DIR}' — run the training cell first.")


def _gaussian_ellipse(mean, cov, ax, n_std=2.0, **kwargs):
    """Draw an n_std confidence ellipse of a 2D Gaussian (mean, cov) on ax.

    Also draws the major/minor axes as lines through the center.
    Returns (ellipse_patch, major_axis_unit_vector).
    """
    mean = np.asarray(mean)
    vals, vecs = np.linalg.eigh(cov)
    order = vals.argsort()[::-1]
    vals, vecs = vals[order], vecs[:, order]

    angle = np.degrees(np.arctan2(vecs[1, 0], vecs[0, 0]))
    semi_axes = n_std * np.sqrt(np.clip(vals, 0, None))
    width, height = 2 * semi_axes

    ell = Ellipse(xy=mean, width=width, height=height, angle=angle,
                   fill=False, **kwargs)
    ax.add_patch(ell)

    for i in range(2):
        d = vecs[:, i] * semi_axes[i]
        p0, p1 = mean - d, mean + d
        ax.plot([p0[0], p1[0]], [p0[1], p1[1]], color="#FF00FF",
                 linewidth=1.5, linestyle="-", alpha=0.9, zorder=10)

    major_axis_unit = vecs[:, 0]
    return ell, major_axis_unit


def _residuals_at(x_val, y_val, t_val):
    pt_np = np.array([[x_val, y_val]])
    v_true = true_velocity(pt_np, t_val)[0]  # (2,)

    pt = torch.tensor(pt_np, dtype=torch.float32, device=device)
    tt = torch.full((1, 1), t_val, dtype=torch.float32, device=device)

    residuals = []
    with torch.no_grad():
        for net in _nets:
            v_pred = net(pt, tt)[0].cpu().numpy()
            residuals.append(v_pred - v_true)
    return np.array(residuals)  # (N, 2)


def _learned_velocity_mean(x_val, y_val, t_val):
    """Ensemble-mean prediction of the learned velocity field at (x, y, t)."""
    pt = torch.tensor([[x_val, y_val]], dtype=torch.float32, device=device)
    tt = torch.full((1, 1), t_val, dtype=torch.float32, device=device)
    preds = []
    with torch.no_grad():
        for net in _nets:
            preds.append(net(pt, tt)[0].cpu().numpy())
    return np.mean(preds, axis=0)


def _learned_velocity_jacobian(x_val, y_val, t_val, h=1e-4):
    """Finite-difference Jacobian of the ensemble-mean learned velocity field."""
    base = np.array([x_val, y_val])
    J = np.zeros((2, 2))
    for i in range(2):
        dp = np.zeros(2)
        dp[i] = h
        v_plus = _learned_velocity_mean(*(base + dp), t_val)
        v_minus = _learned_velocity_mean(*(base - dp), t_val)
        J[:, i] = (v_plus - v_minus) / (2 * h)
    return J


def _true_velocity_jacobian(x_val, y_val, t_val, h=1e-4):
    """Finite-difference Jacobian (d v_true / d x) of the true velocity field at (x, y, t)."""
    base = np.array([x_val, y_val])
    J = np.zeros((2, 2))
    for i in range(2):
        dp = np.zeros(2)
        dp[i] = h
        v_plus = true_velocity((base + dp)[None, :], t_val)[0]
        v_minus = true_velocity((base - dp)[None, :], t_val)[0]
        J[:, i] = (v_plus - v_minus) / (2 * h)
    return J


def _whitening_matrix(J):
    """(J^T J)^-1/2, used to whiten residuals against the local sensitivity scale of v(x,t).

    residual ~ J @ delta for some upstream ~isotropic perturbation delta, so
    Cov(residual) ~ J @ Cov(delta) @ J^T ~ J @ J^T. Whitening needs M with
    M @ (J J^T) @ M^T = I, i.e. M = (J^T J)^-1/2 (the unique symmetric PSD square
    root inverse — this is just the polar/SVD whitening transform). J^T J is always
    symmetric PSD regardless of any asymmetry in J itself (e.g. from finite-difference
    noise), so eigh on it (rather than on J directly) is the numerically correct move.
    """
    A = J.T @ J
    vals, vecs = np.linalg.eigh(A)
    vals = np.clip(vals, 1e-12, None)
    inv_sqrt = 1.0 / np.sqrt(vals)
    return vecs @ np.diag(inv_sqrt) @ vecs.T


def _density_pt(Gx, Gy, t_val):
    """Density of the t-marginal of the circle flow (uniform-on-circle + Gaussian noise)."""
    r = np.sqrt(Gx**2 + Gy**2)
    s = max(1 - t_val, 1e-6)
    kappa = t_val * r / s**2
    log_p = np.log(ive(0, kappa) + 1e-300) + kappa - (r**2 + t_val**2) / (2 * s**2)
    p = np.exp(log_p - log_p.max())
    return p


lim = 1.5
theta_c = np.linspace(0, 2 * np.pi, 300)

res_d = 200
gx_d = np.linspace(-lim, lim, res_d)
gy_d = np.linspace(-lim, lim, res_d)
Gx_d, Gy_d = np.meshgrid(gx_d, gy_d)

state = {"x": 0.7, "y": 0.3}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.5))
fig.canvas.toolbar_visible = False


def _draw_background(ax, x_val, y_val, t_val):
    density = _density_pt(Gx_d, Gy_d, t_val)
    ax.pcolormesh(Gx_d, Gy_d, density, shading="auto", cmap="Blues", alpha=0.5, zorder=0)
    ax.plot(np.cos(theta_c), np.sin(theta_c), "k--", linewidth=1, label="unit circle")
    ax.plot(x_val, y_val, "g*", markersize=14, label=f"point=({x_val:.2f}, {y_val:.2f})")


def _finish_axes(ax, title):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect("equal")
    ax.set_title(title)
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=0.3)


def redraw(*_):
    t_val = t_slider.value
    n_std = scale_slider.value
    whitened_n_std = whitened_scale_slider.value
    x_val, y_val = state["x"], state["y"]
    point = np.array([x_val, y_val])

    residuals = _residuals_at(x_val, y_val, t_val) * (1 - t_val)
    mean = residuals.mean(axis=0)
    cov = np.cov(residuals.T)

    if jacobian_source_toggle.value == "true":
        J = _true_velocity_jacobian(x_val, y_val, t_val)
    else:
        J = _learned_velocity_jacobian(x_val, y_val, t_val)
    J_eigvals = np.linalg.eigvalsh(J)
    ratio = np.abs(J_eigvals).max() / np.abs(J_eigvals).min()
    eig_output.clear_output(wait=True)
    with eig_output:
        print(f"point=({x_val:.3f}, {y_val:.3f})  t={t_val:.3f}  J source={jacobian_source_toggle.value}")
        print(f"J eigenvalues: {J_eigvals}  |ratio|={ratio:.3f}")
    M = _whitening_matrix(J)
    whitened = residuals @ M.T
    whitened_mean = whitened.mean(axis=0)
    whitened_cov = np.cov(whitened.T)

    radius_norm = np.linalg.norm(point)
    radius_unit = point / radius_norm if radius_norm > 1e-9 else None

    def _angle_to_radius(major_axis_unit):
        if radius_unit is None:
            return float("nan")
        cos_angle = np.clip(abs(np.dot(major_axis_unit, radius_unit)), 0.0, 1.0)
        return np.degrees(np.arccos(cos_angle))

    # --- left: raw residuals ---
    ax1.cla()
    _draw_background(ax1, x_val, y_val, t_val)

    offsets = point + residuals
    ax1.scatter(offsets[:, 0], offsets[:, 1], s=20, alpha=0.6, label="residuals (offset from point)")
    mean_xy = point + mean
    ax1.scatter(*mean_xy, color="red", marker="x", s=80, label="mean residual")
    _, major_axis_unit = _gaussian_ellipse(mean_xy, cov, ax1, n_std=n_std, color="red",
                                            linestyle="--", linewidth=1.5, label=f"{n_std:.1f}σ fit")
    angle_deg = _angle_to_radius(major_axis_unit)

    _finish_axes(ax1, f"Raw residuals  t={t_val:.3f}\n"
                      f"|angle(major axis, radius)| = {angle_deg:.1f}°\n"
                      f"(click anywhere to move evaluation point)")

    # --- right: whitened residuals ---
    ax2.cla()
    _draw_background(ax2, x_val, y_val, t_val)

    whitened_offsets = point + whitened
    ax2.scatter(whitened_offsets[:, 0], whitened_offsets[:, 1], s=40, alpha=0.6,
                color="orange", edgecolor="black", linewidth=0.5, zorder=15,
                label="whitened residuals ((J^T J)^-1/2 r)")
    whitened_mean_xy = point + whitened_mean
    ax2.scatter(*whitened_mean_xy, color="darkorange", marker="x", s=80, zorder=16,
                label="whitened mean")
    _, whitened_major_axis_unit = _gaussian_ellipse(whitened_mean_xy, whitened_cov, ax2, n_std=whitened_n_std,
                                                      color="orange", linestyle="--", linewidth=1.5,
                                                      label=f"{whitened_n_std:.1f}σ fit")
    whitened_angle_deg = _angle_to_radius(whitened_major_axis_unit)

    _finish_axes(ax2, f"Whitened residuals ((J^T J)^-1/2 r)  t={t_val:.3f}\n"
                      f"|angle(major axis, radius)| = {whitened_angle_deg:.1f}°\n"
                      f"(click anywhere to move evaluation point)")

    fig.tight_layout()
    fig.canvas.draw_idle()


def on_click(event):
    if event.inaxes not in (ax1, ax2) or event.xdata is None:
        return
    state["x"], state["y"] = event.xdata, event.ydata
    redraw()


fig.canvas.mpl_connect("button_press_event", on_click)

t_slider = widgets.FloatSlider(value=0.8, min=0.0, max=0.999, step=0.01, description="t")
scale_slider = widgets.FloatSlider(value=2.0, min=0.5, max=5.0, step=0.1, description="ellipse n_std")
whitened_scale_slider = widgets.FloatSlider(value=5.0, min=0.5, max=10.0, step=0.1, description="whitened n_std")
jacobian_source_toggle = widgets.ToggleButtons(options=["true", "learned"], value="true",
                                                description="J source")

t_slider.observe(redraw, names="value")
scale_slider.observe(redraw, names="value")
whitened_scale_slider.observe(redraw, names="value")
jacobian_source_toggle.observe(redraw, names="value")

eig_output = widgets.Output()

display(VBox([HBox([t_slider, scale_slider, whitened_scale_slider]), jacobian_source_toggle,
               eig_output, fig.canvas]))
redraw()